# Embeddings, vector stores and retrievers

Assistant version 4: Search waterproof four-person tents under $300 with category and price constraints.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra rag --group authoring`, then `uv run --extra rag --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Embedding meaning; numpy cosine similarity by hand; 2D projection with caveats

Predict the output before running. The function definitions below match `examples/01_cosine_by_hand.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Calculate cosine similarity and show a labelled two-dimensional projection.

Run: uv run --extra rag python modules/07_embeddings_retrievers/examples/01_cosine_by_hand.py
--offline
Expected: Cosine similarity near 1 for aligned vectors, and an illustrative projection PNG.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

import numpy as np

from common.config import ROOT
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        raise ValueError("Cosine is undefined for a zero vector.")
    return float(np.dot(a, b) / denominator)


def main() -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    settings = lesson_settings()
    vectors = np.array([[1.0, 0.1, 0.0], [0.9, 0.2, 0.0], [0.0, 0.2, 1.0], [0.1, 0.0, 0.9]])
    print_text(f"Illustrative tent similarity: {cosine(vectors[0], vectors[1]):.3f}")
    centered = vectors - vectors.mean(axis=0)
    _, _, axes = np.linalg.svd(centered, full_matrices=False)
    projected = centered @ axes[:2].T
    # KEY IDEA: these vectors are invented teaching coordinates, not learned embeddings.
    figure, axis = plt.subplots(figsize=(7, 4))
    axis.scatter(projected[:, 0], projected[:, 1])
    axis.margins(0.2)
    for label, point in zip(["tent A", "tent B", "boot A", "boot B"], projected, strict=True):
        axis.annotate(label, point, xytext=(5, 5), textcoords="offset points")
    axis.set_title("Illustrative coordinates projected to 2D; distances can distort")
    output = ROOT / settings.runtime_dir / "embedding_projection.png"
    output.parent.mkdir(parents=True, exist_ok=True)
    figure.tight_layout()
    figure.savefig(output, dpi=150)
    plt.close(figure)
    print_text(str(output))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## embed_query, embed_documents; hosted and local models; separate embedding provider

Predict the output before running. The function definitions below match `examples/02_embedding_interface.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Use independent embedding configuration through embed_query and embed_documents.

Run: uv run --extra rag python
modules/07_embeddings_retrievers/examples/02_embedding_interface.py --offline
Expected: Two document vectors and a query vector with matching dimensions.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_embeddings
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    embeddings = get_embeddings(settings=settings)
    documents = embeddings.embed_documents(
        ["Waterproof Brightwave tent", "Brightwave hiking boots"]
    )
    query = embeddings.embed_query("shelter for a rainy trip")
    # KEY IDEA: query/document APIs may use different task prefixes inside a provider.
    print_text(f"Document vectors={len(documents)}, dimensions={len(query)}")
    print_text(
        "Fake embeddings are repeatable random vectors, not semantic representations."
        if settings.course_offline
        else f"Embedding provider: {settings.embedding_provider}; model: {settings.embedding_model}"
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## InMemoryVectorStore; add documents and similarity_search

Predict the output before running. The function definitions below match `examples/03_memory_store.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Add documents to an in-memory vector index and retrieve similar text.

Run: uv run --extra rag python modules/07_embeddings_retrievers/examples/03_memory_store.py
--offline
Expected: Three result IDs and sources; offline order has no semantic meaning.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import memory_index


def main() -> None:
    # KEY IDEA: The query and stored documents must use the same embedding space.
    settings = lesson_settings()
    documents = chunk_documents(policy_documents())
    store = memory_index(documents, settings)
    for doc in store.similarity_search("How long do returns take?", k=3):
        print_text(str(doc.metadata))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Chroma persistence; collection identity; model/dimension metadata

Predict the output before running. The function definitions below match `examples/04_persistent_chroma.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Reopen a local Chroma collection only when its manifest matches.

Run: uv run --extra rag python
modules/07_embeddings_retrievers/examples/04_persistent_chroma.py --offline
Expected: The same stored document IDs before and after reopening.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.vector_index import persistent_index


def main() -> None:
    settings = lesson_settings()
    documents = chunk_documents(policy_documents())
    first = persistent_index(documents, settings, name="lesson07-persistence")
    second = persistent_index(documents, settings, name="lesson07-persistence")
    # KEY IDEA: persistence is automatic; model/dimension/corpus identity is our responsibility.
    assert set(first.get()["ids"]) == set(second.get()["ids"])
    print_text(f"Reopened {len(second.get()['ids'])} documents without re-adding the corpus.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Add, update and delete documents by ID; rebuild incompatible indexes

Predict the output before running. The function definitions below match `examples/05_document_lifecycle.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Add, update and delete a temporary document by a stable explicit ID.

Run: uv run --extra rag python
modules/07_embeddings_retrievers/examples/05_document_lifecycle.py --offline
Expected: A temporary note changes content, then disappears; seed documents stay intact.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from langchain_core.documents import Document

from common.errors import run_cli
from common.knowledge import chunk_documents, policy_documents
from common.lesson import lesson_settings
from common.pretty import print_text
from common.vector_index import persistent_index


def main() -> None:
    # KEY IDEA: Stable IDs select which record is updated or deleted.
    settings = lesson_settings()
    store = persistent_index(
        chunk_documents(policy_documents()), settings, name="lesson07-lifecycle"
    )
    key = "temporary-training-note"
    store.add_documents(
        [Document(page_content="Brightwave demo stock: 10", metadata={"source": "synthetic-note"})],
        ids=[key],
    )
    store.update_documents(
        [key],
        [Document(page_content="Brightwave demo stock: 8", metadata={"source": "synthetic-note"})],
    )
    print_text(str(store.get(ids=[key])["documents"]))
    store.delete(ids=[key])
    assert store.get(ids=[key])["ids"] == []
    print_text("Temporary ID deleted. Changed embedding/corpus identity requires a new index name.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Similarity with scores; MMR; metadata filters; distinguish scores from probabilities

Predict the output before running. The function definitions below match `examples/06_scores_mmr_filters.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare scored similarity with diverse MMR results under a metadata filter.

Run: uv run --extra rag python
modules/07_embeddings_retrievers/examples/06_scores_mmr_filters.py --offline
Expected: Only tents appear; cosine distances are not answer probabilities.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import product_documents
from common.vector_index import persistent_index


def main() -> None:
    settings = lesson_settings()
    store = persistent_index(product_documents(), settings, name="lesson07-products")
    where = {"$and": [{"category": "tents"}, {"price": {"$lt": 300}}]}
    for doc, distance in store.similarity_search_with_score("waterproof tent", k=3, filter=where):
        print_text(f"{doc.metadata['sku']}: cosine distance={distance:.4f}; lower is closer")
    # KEY IDEA: MMR trades off query similarity against repeated, similar results.
    diverse = store.max_marginal_relevance_search(
        "waterproof tent", k=3, fetch_k=8, lambda_mult=0.5, filter=where
    )
    print_text("MMR: " + str([doc.metadata["sku"] for doc in diverse]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## as_retriever; retrieval configuration

Predict the output before running. The function definitions below match `examples/07_retriever.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Expose a vector store as a configurable Runnable retriever.

Run: uv run --extra rag python modules/07_embeddings_retrievers/examples/07_retriever.py
--offline
Expected: Two filtered tent Documents from retriever.invoke.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import product_documents
from common.vector_index import persistent_index


def main() -> None:
    settings = lesson_settings()
    store = persistent_index(product_documents(), settings, name="lesson07-products")
    retriever = store.as_retriever(
        search_type="similarity", search_kwargs={"k": 2, "filter": {"category": "tents"}}
    )
    # KEY IDEA: a retriever returns Documents; generation belongs to a separate stage.
    print_text(str([doc.metadata["sku"] for doc in retriever.invoke("rain shelter")]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Build a small Embeddings wrapper cache; key by provider, model, dimensions and text

Predict the output before running. The function definitions below match `examples/08_embedding_cache.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Cache vectors by model identity, dimension, input text and query/document task.

Run: uv run --extra rag python modules/07_embeddings_retrievers/examples/08_embedding_cache.py
--offline
Expected: Repeated query vectors match; cache survives another wrapper instance.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.config import ROOT
from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_embeddings
from common.pretty import print_text
from common.vector_index import CachedEmbeddings, embedding_identity


def main() -> None:
    # KEY IDEA: Reuse vectors only when text, operation kind and embedding identity all match.
    settings = lesson_settings()
    wrapped = get_embeddings(settings=settings)
    dimensions = len(wrapped.embed_query("dimension probe"))
    identity = embedding_identity(settings, dimensions)
    cache = CachedEmbeddings(wrapped, ROOT / settings.runtime_dir / "embeddings.sqlite", identity)
    first = cache.embed_query("Brightwave waterproof tent")
    second = cache.embed_query("Brightwave waterproof tent")
    assert first == second
    print_text(f"Cached vector dimensions={len(first)}; identity={identity}")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Search waterproof four-person tents under $300 with category and price constraints

Predict the output before running. The function definitions below match `use_case/product_search.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Enforce catalogue constraints before ranking waterproof four-person tents under $300.

Run: uv run --extra rag python modules/07_embeddings_retrievers/use_case/product_search.py
--offline
Expected: Up to three matching tents, each strictly under $300 and exactly four-person.
Cost: Offline: no API fees. Live embeddings depend on corpus size and the chosen provider.
Requires: rag
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.retrieval import product_search


def main() -> None:
    settings = lesson_settings()
    for doc in product_search(settings):
        print_text(str(doc.metadata))
    print_text("Constraints are real. Offline ranking uses non-semantic fake vectors.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra rag pytest modules/07_embeddings_retrievers/tests` for the supplied solution checks.